# Student Dropout Prediction Project

## Notebook 02: Data Preprocessing

### Objective

This notebook prepares the student dropout dataset for machine learning.

Two prediction stages are maintained:

1. **Enrollment-only prediction**
   Uses information available when a student enrolls.

2. **Semester 1 prediction**
   Uses enrollment information together with first-semester academic information.

The target variable is `Target`, containing three classes:

- Dropout
- Enrolled
- Graduate

The preprocessing includes separating features from the target, identifying categorical and numerical variables, splitting the data into training and testing sets, and creating preprocessing pipelines for machine learning.

In [34]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [35]:
enrollment_df = pd.read_csv("../data/processed/enrollment_only.csv")
semester1_df = pd.read_csv("../data/processed/semester1.csv")

print("Enrollment-only shape:", enrollment_df.shape)
print("Semester 1 shape:", semester1_df.shape)

Enrollment-only shape: (4424, 25)
Semester 1 shape: (4424, 31)


In [36]:
X_enrollment = enrollment_df.drop(columns=["Target"])
y_enrollment = enrollment_df["Target"]

X_semester1 = semester1_df.drop(columns=["Target"])
y_semester1 = semester1_df["Target"]

In [37]:
print("Enrollment-only:")
print("X:", X_enrollment.shape)
print("y:", y_enrollment.shape)

print("\nSemester 1:")
print("X:", X_semester1.shape)
print("y:", y_semester1.shape)

Enrollment-only:
X: (4424, 24)
y: (4424,)

Semester 1:
X: (4424, 30)
y: (4424,)


In [38]:
categorical_cols = [
    "Marital Status",
    "Application mode",
    "Application order",
    "Course",
    "Daytime/evening attendance",
    "Previous qualification",
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
    "Displaced",
    "Educational special needs",
    "Debtor",
    "Tuition fees up to date",
    "Gender",
    "Scholarship holder",
    "International"
]

In [39]:
numerical_cols = [
    "Previous qualification (grade)",
    "Admission grade",
    "Age at enrollment"
]

In [40]:
semester1_numerical_cols = numerical_cols + [
    "Curricular units 1st sem (credited)",
    "Curricular units 1st sem (enrolled)",
    "Curricular units 1st sem (evaluations)",
    "Curricular units 1st sem (approved)",
    "Curricular units 1st sem (grade)",
    "Curricular units 1st sem (without evaluations)"
]

In [41]:
X_train_enrollment, X_test_enrollment, y_train_enrollment, y_test_enrollment = train_test_split(
    X_enrollment,
    y_enrollment,
    test_size=0.20,
    random_state=42,
    stratify=y_enrollment
)

In [42]:
X_train_semester1, X_test_semester1, y_train_semester1, y_test_semester1 = train_test_split(
    X_semester1,
    y_semester1,
    test_size=0.20,
    random_state=42,
    stratify=y_semester1
)

In [43]:
categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

numerical_transformer = StandardScaler()

In [44]:
enrollment_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_cols),
        ("num", numerical_transformer, numerical_cols)
    ]
)

In [45]:
semester1_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_cols),
        ("num", numerical_transformer, semester1_numerical_cols)
    ]
)

In [46]:
X_train_enrollment_processed = enrollment_preprocessor.fit_transform(
    X_train_enrollment
)

X_test_enrollment_processed = enrollment_preprocessor.transform(
    X_test_enrollment
)

In [47]:
X_train_semester1_processed = semester1_preprocessor.fit_transform(
    X_train_semester1
)

X_test_semester1_processed = semester1_preprocessor.transform(
    X_test_semester1
)

In [48]:
print("Enrollment-only")
print("Original features:", X_enrollment.shape[1])
print("Processed features:", X_train_enrollment_processed.shape[1])

print("\nSemester 1")
print("Original features:", X_semester1.shape[1])
print("Processed features:", X_train_semester1_processed.shape[1])

Enrollment-only
Original features: 24
Processed features: 217

Semester 1
Original features: 30
Processed features: 223


In [49]:
print("Training target distribution:")
print(y_train_enrollment.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test_enrollment.value_counts(normalize=True))

Training target distribution:
Target
Graduate    0.499294
Dropout     0.321277
Enrolled    0.179429
Name: proportion, dtype: float64

Testing target distribution:
Target
Graduate    0.499435
Dropout     0.320904
Enrolled    0.179661
Name: proportion, dtype: float64
